# Single-frame ResNet50 steering (spatial 9728 features)

Run the cells in order from a checkout containing the seeded indices in the directory configured by `road-seed-splits.json` and their archived images. The first run needs the pretrained MyDeZero ResNet50 weights. Each execution creates its own config, checkpoints and evaluation files. Inference uses only the current RGB image. The primary endpoint is frame-weighted MAE against unmodified PID commands, compared with zero steering on exactly the same frames.

## Training recipes

| Recipe | Training label on spike frames | Spike weight | Large non-spike weight |
| --- | --- | ---: | ---: |
| `raw` | `raw` | 1 | 1 |
| `raw_x2` | `raw` | 1 | 2 |
| `raw_x4` | `raw` | 1 | 4 |
| `downweighted_x1` | `raw` | 0.1 | 1 |
| `downweighted_x2` | `raw` | 0.1 | 2 |
| `downweighted_x4` | `raw` | 0.1 | 4 |
| `smoothed_x1` | `target` | 1 | 1 |
| `smoothed_x2` | `target` | 1 | 2 |
| `smoothed_x4` | `target` | 1 | 4 |


On isolated spikes, `target` is the mean of the preceding and following raw commands. All other frames retain `raw`. A large turn has absolute raw steering of at least 0.3. Non-spike frames below this threshold have weight 1.

In [ ]:
from donkeycar_app.checkpoints import checkpoint_recipe, export_best_checkpoint
from donkeycar_app.dataset import STEERING_RECIPES
import json
from pathlib import Path
import numpy as np
from PIL import Image

from mydezero import cuda, no_grad, test_mode
import mydezero.functions as F
from mydezero.layers import Linear
from mydezero.models import ResNet50, VGG16


class SpatialSteeringResNet(ResNet50):
    """Predict normalized steering from mean-centered BGR frames (N, 3, 120, 160)."""

    def __init__(self, pretrained=False):
        super().__init__(pretrained=pretrained)
        self.fc6 = Linear(1, in_size=512 * 3 * 5 + 2048)

    def frozen_features(self, images):
        with test_mode(), no_grad():
            x = F.pooling(F.relu(self.bn1(self.conv1(images))), 3, stride=2)
            mid = self.res3(self.res2(x))
            top = self.res5(self.res4(mid))
            local = mid.data.reshape(len(images), 512, 3, 5, 5, 4).mean(axis=(3, 5))
            global_features = top.data.mean(axis=(2, 3))
            xp = cuda.get_array_module(global_features)
            return xp.concatenate((local.reshape(len(images), -1), global_features), axis=1)

    def forward(self, images):
        return F.clip(self.fc6(self.frozen_features(images)), -1, 1)

    def steer(self, rgb_frame):
        """Predict one steering command from a 120 by 160 RGB camera frame."""
        frame = np.asarray(rgb_frame)
        if frame.shape != (120, 160, 3) or frame.dtype != np.uint8:
            raise ValueError("Expected a uint8 RGB camera frame of shape (120, 160, 3)")
        pixels = VGG16.preprocess(Image.fromarray(frame), size=None)
        xp = cuda.get_array_module(self.fc6.W.data)
        with test_mode(), no_grad():
            return float(self(xp.asarray(np.ascontiguousarray(pixels[None]))).data.item())

NOTEBOOK_NAME = "resnet50_single_frame_spatial_9728"


def load_steering_model(checkpoint, use_gpu=True, *, recipe=None):
    checkpoint = Path(checkpoint)
    recipe = checkpoint_recipe(checkpoint, recipe)
    model = SpatialSteeringResNet()
    model.load_weights(checkpoint)
    if model.fc6.W.shape != (9728, 1):
        raise ValueError("Expected a 9728-to-1 steering head")
    if use_gpu and cuda.gpu_enable and cuda.cupy.is_available():
        model.to_gpu()
    multipliers = {"downweighted": 1, "interpolated": 1, "sharp_x2": 2, "sharp_x4": 4}  # Older checkpoints
    multipliers.update({name: spec.sharp_multiplier for name, spec in STEERING_RECIPES.items()})
    if recipe not in multipliers:
        raise ValueError(f"Specify a known checkpoint recipe: {list(multipliers)}")
    model.drive_log_prefix = f"{NOTEBOOK_NAME}-{multipliers[recipe]}x-{recipe}"
    return model

## Drive a saved checkpoint

Start DonkeySim, run the model definition above, then set `drive_saved_model = True` and run the cell below. Choose a listed checkpoint by number or enter a .npz path. Set `recipe` if the checkpoint was renamed or a standalone `best.npz` has no training report. Leave driving disabled for training.

In [ ]:
drive_saved_model = False
if drive_saved_model:
    from donkeycar_app.drive import run_drive, select_checkpoint

    checkpoint = select_checkpoint(NOTEBOOK_NAME)
    recipe = None
    driving_model = load_steering_model(checkpoint, recipe=recipe)
    run_drive(driving_model.steer)

In [ ]:
from datetime import datetime, timezone
import csv
import hashlib
import json
from pathlib import Path
from uuid import uuid4

import numpy as np
from PIL import Image

from mydezero import cuda, no_grad, test_mode
import mydezero.functions as F
from mydezero.layers import Linear
from mydezero.optimizers import Adam
from donkeycar_app.dataset import preprocess_camera, read_split, steering_loss_weights

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/mydezero").is_dir())
seed = 42
np.random.seed(seed)
use_gpu = cuda.gpu_enable and cuda.cupy.cuda.runtime.getDeviceCount() > 0
xp = cuda.cupy if use_gpu else np
run = NOTEBOOK_NAME + "-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:6]
out = root / "runs/donkeycar" / run
weights_dir = root / "models/donkeycar" / run
manifest = root / "examples/donkeycar/road-seed-splits.json"
data_config = json.loads(manifest.read_text(encoding="utf-8"))
index_dir = root / data_config["index_dir"]
for folder in (out, weights_dir):
    folder.mkdir(parents=True)
def save_json(path, value):
    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\n", encoding="utf-8")

print(run, "CUDA" if use_gpu else "CPU")

## Load the seeded train, validation and test splits

In [ ]:
indices = {split: index_dir / f"{split}.json" for split in ("train", "validation", "test")}
rows = {split: read_split(path) for split, path in indices.items()}
seeds = {split: {r["seed"] for r in part} for split, part in rows.items()}
assert sum(map(len, seeds.values())) == len(set.union(*seeds.values()))
for split, part in rows.items():
    repaired = sum(r["raw"] != r["target"] for r in part)
    print(split, len(part), "frames,", repaired, "repaired labels")
    if split != "train":
        assert repaired == 0

## Extract frozen features

Pool the third residual stage on a 3 × 5 grid to preserve the position of road markings. Append global features from the final stage. Standardization statistics are fitted only on training data. The backbone uses pretrained weights in evaluation mode.

In [ ]:
model = SpatialSteeringResNet(pretrained=True)
if use_gpu:
    model.to_gpu()
def pixels(row):
    with Image.open(root / row["image"]) as im:
        if im.mode != "RGB" or im.size != (160, 120):
            raise ValueError(row["image"])
        return preprocess_camera(im)

def features(part, model):
    result = []
    for start in range(0, len(part), 32):
        batch = np.ascontiguousarray(np.stack([pixels(r) for r in part[start:start + 32]]))
        result.append(cuda.as_numpy(model.frozen_features(xp.asarray(batch))))
    return xp.asarray(np.concatenate(result))

def column(part, key):
    return xp.asarray([r[key] for r in part], dtype=xp.float32).reshape(-1, 1)

fit, val = rows["train"], rows["validation"]
X = features(fit, model)
V = features(val, model)
center = X.mean(axis=0)
scale = xp.maximum(X.std(axis=0), 0.1)
X -= center
X /= scale
V -= center
V /= scale
print(X.shape, V.shape)

## Compare steering loss weights

Use a smooth absolute-error loss with transition scale 0.05. All heads start at zero and use the same minibatch sequence and learning rate schedule. Cross each of the three label and spike-weight settings with 1x, 2x and 4x weights for original commands with absolute magnitude at least 0.3, excluding flagged reverse spikes. Large commands are a proxy for sharp turns. The multiplier applies only to training weights, and each batch loss is divided by its weight sum.

Evaluate every epoch against original validation labels with unit weights. Track overall MAE, large commands without flagged spikes, ordinary commands and flagged spikes. Save each recipe's best checkpoint by overall validation MAE and export the overall winner as best.npz. Compare subgroup errors before choosing a candidate for closed-loop driving.


In [ ]:
val_y = column(val, "raw")
epochs, batch_size, delta = 24, 256, 0.05
sharp_threshold = 0.3
val_spikes = np.array([r["spike"] for r in val], dtype=bool)
val_large = np.abs(cuda.as_numpy(val_y).ravel()) >= sharp_threshold
val_groups = {
    "sharp_nonspike": xp.asarray(val_large & ~val_spikes),
    "ordinary": xp.asarray(~val_large),
    "spikes": xp.asarray(val_spikes),
}
save_json(out / "config.json", dict(seed=seed, epochs=epochs, feature_batch_size=batch_size, delta=delta,
          indices={p.relative_to(root).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in indices.values()},
          seeds={k: sorted(v) for k, v in seeds.items()}, sharp_threshold=sharp_threshold,
          recipes={name: spec._asdict() for name, spec in STEERING_RECIPES.items()}, spike_weight=0.1, normalization="batch_weight_sum",
          selection="overall_validation_mae"))
history = []
selections = {}
best_mae = float("inf")
for recipe, spec in STEERING_RECIPES.items():
    rng = np.random.default_rng(seed)
    head = Linear(1, in_size=X.shape[1])
    head.W.data.fill(0)
    if use_gpu:
        head.to_gpu()
    optim = Adam(lr=0.00008).setup(head)
    y = column(fit, spec.label)
    w = xp.asarray(steering_loss_weights(fit, spec.sharp_multiplier, sharp_threshold,
                                        downweight_spikes=spec.downweight_spikes))
    for epoch in range(1, epochs + 1):
        optim.lr = 0.00008 * (0.5 ** ((epoch - 1) // 8))
        order = rng.permutation(len(fit))
        for i in range(0, len(order), batch_size):
            idx = order[i:i + batch_size]
            residual = head(X[idx]) - y[idx]
            loss = F.sum(((residual ** 2 + delta ** 2) ** 0.5 - delta) * w[idx]) / float(w[idx].sum())
            head.cleargrads()
            loss.backward()
            optim.update()
        with no_grad():
            pred = xp.clip(head(V).data, -1, 1)
            mae = float(xp.abs(pred - val_y).mean())
        group_mae = {name: float(xp.abs(pred - val_y).ravel()[mask].mean())
                     for name, mask in val_groups.items() if bool(mask.any())}
        history.append(dict(recipe=recipe, epoch=epoch, validation_mae=mae, **group_mae))
        with (out / f"{recipe}-epochs.jsonl").open("a", encoding="utf-8") as log:
            log.write(json.dumps(dict(epoch=epoch, validation=dict(mae=mae, **group_mae), learning_rate=optim.lr), allow_nan=False) + "\n")
        if recipe not in selections or mae < selections[recipe]["mae"]:
            candidate_path = weights_dir / f"{recipe}.npz"
            model.fc6.W.data = head.W.data / scale[:, None]
            model.fc6.b.data = head.b.data - center @ model.fc6.W.data
            model.save_weights(candidate_path)
            selections[recipe] = dict(epoch=epoch, mae=mae, **group_mae,
                                      checkpoint=candidate_path.relative_to(root).as_posix())
        if mae < best_mae:
            best_mae, best_recipe, best_epoch = mae, recipe, epoch
            best_w, best_b = head.W.data.copy(), head.b.data.copy()
    print(recipe, selections[recipe], flush=True)
save_json(out / "history.json", history)
val_baseline = float(xp.abs(val_y).mean())
print("Selected", best_recipe, best_epoch, best_mae, "zero", val_baseline)

## Export and check the selected model

Fold training-set standardization into the regression weights. The checkpoint contains the full backbone and readout. Validate single-image inference after reloading on CPU against the original validation predictions.

In [ ]:
reference = cuda.as_numpy(xp.clip(V[:4] @ best_w + best_b, -1, 1)).ravel()
checkpoint = export_best_checkpoint(weights_dir, best_recipe)
inference = SpatialSteeringResNet()
inference.load_weights(checkpoint)
def rgb(row):
    with Image.open(root / row["image"]) as im:
        return np.array(im)
actual = [inference.steer(rgb(r)) for r in val[:4]]
np.testing.assert_allclose(actual, reference, atol=3e-5, rtol=1e-4)
if use_gpu:
    inference.to_gpu()
print("CPU checkpoint round trip passed", checkpoint.relative_to(root))

## Evaluate on the original held-out drives

Use all frames, including spikes. Report frame-weighted errors on the same PID targets for both predictors. The sustained-turn subset requires three consecutive commands with the same sign and magnitude at least 0.3. These drives were used in previous project evaluations. They are a historical holdout, not a fresh blind test.

In [ ]:
def scores(part, predictions):
    targets = np.array([r["raw"] for r in part])
    err = np.abs(predictions - targets)
    zero = np.abs(targets)
    groups = {"all": np.ones(len(part), dtype=bool),
              "sharp": zero >= 0.3,
              "sustained": np.array([r["sustained"] for r in part]),
              "spikes": np.array([r["spike"] for r in part]),
              "sharp_nonspike": (zero >= sharp_threshold) & ~np.array([r["spike"] for r in part]),
              "ordinary": zero < sharp_threshold}
    for route in sorted({r["route"] for r in part}):
        groups[route] = np.array([r["route"] == route for r in part])
    return {k: dict(frames=int(mask.sum()), mae=float(err[mask].mean()),
                    zero_mae=float(zero[mask].mean())) for k, mask in groups.items() if mask.any()}

report = dict(seed=seed, run=run, checkpoint=checkpoint.relative_to(root).as_posix(),
              validation_candidates=selections, selected_recipe=best_recipe, epoch=best_epoch,
              index=index_dir.relative_to(root).as_posix(), device="CUDA" if use_gpu else "CPU")
for name in ("validation", "test"):
    part = rows[name]
    with no_grad(), test_mode():
        if name == "validation":
            pred = cuda.as_numpy(xp.clip(V @ best_w + best_b, -1, 1)).ravel()
        else:
            test_features = features(part, inference)
            pred = cuda.as_numpy(F.clip(inference.fc6(test_features), -1, 1)).ravel()
    assert np.isfinite(pred).all() and len(pred) == len(part)
    report[name] = scores(part, pred)
    with (out / f"{name}-predictions.csv").open("w", newline="", encoding="utf-8") as file:
        writer = csv.writer(file)
        writer.writerow(("route", "frame", "target", "prediction"))
        writer.writerows((r["route"], r["frame"], r["raw"], float(p)) for r, p in zip(part, pred))
save_json(out / "metrics.json", report)
print(json.dumps(report, indent=2))

def measure(prediction, target):
    if not np.isfinite(prediction).all():
        raise ValueError("Non-finite steering prediction")
    errors = np.abs(prediction - target)
    sharp = np.abs(target) >= sharp_threshold
    return {
        "samples": int(target.size),
        "mae": float(errors.mean()),
        "sharp_samples": int(sharp.sum()),
        "sharp_mae": float(errors[sharp].mean()) if sharp.any() else None,
        "zero_steering_mae": float(np.abs(target).mean()),
    }

test_rows = rows["test"]
test_targets = np.asarray([r["raw"] for r in test_rows], dtype=np.float32).reshape(-1, 1)
route_labels = np.array([row["route"] for row in test_rows])
sharp_nonspike = np.array([abs(r["raw"]) >= sharp_threshold and not r["spike"] for r in test_rows])
results = {}
for recipe, chosen in selections.items():
    candidate = load_steering_model(root / chosen["checkpoint"], use_gpu=use_gpu)
    with no_grad(), test_mode():
        predictions = cuda.as_numpy(F.clip(candidate.fc6(test_features), -1, 1))
    result = dict(checkpoint=chosen["checkpoint"], best_epoch=chosen["epoch"],
                  validation_mae=chosen["mae"], epochs_completed=epochs, stop_reason="epoch_limit")
    results[recipe] = result
    result["overall"] = measure(predictions, test_targets)
    result["sharp_nonspike"] = measure(predictions[sharp_nonspike], test_targets[sharp_nonspike])
    result["by_route"] = {
        route: measure(predictions[route_labels == route], test_targets[route_labels == route])
        for route in sorted(set(route_labels))
    }
(out / "test.json").write_text(json.dumps(results, indent=2, allow_nan=False) + "\n", encoding="utf-8")
print(json.dumps(results, indent=2))

## Load the model for one camera frame

On another machine, install the project, copy the checkpoint, construct `SpatialSteeringResNet()`, and call `load_weights(path)` before `steer(rgb_frame)`. The input is an RGB uint8 array of shape (120, 160, 3). The output is a steering command in [−1, 1].

## Inspect the steering labels and training curves

In [ ]:
import matplotlib.pyplot as plt

example = next((r for r in fit if r["raw"] != r["target"]), fit[0])
segment = [r for r in fit if r["route"] == example["route"] and abs(r["frame"] - example["frame"]) <= 11]
fig, (trace_ax, score_ax) = plt.subplots(1, 2, figsize=(11, 3.6), layout="constrained")
for key, label in (("raw", "Recorded"), ("target", "Interpolated")):
    trace_ax.plot([r["frame"] for r in segment], [r[key] for r in segment], marker=".", label=label)
trace_ax.set(title="Training labels near an isolated spike", xlabel="Training frame", ylabel="Steering command")
trace_ax.legend()
for recipe in STEERING_RECIPES:
    label = recipe
    curve = [r for r in history if r["recipe"] == recipe]
    score_ax.plot([r["epoch"] for r in curve], [r["validation_mae"] for r in curve], label=label)
score_ax.axhline(val_baseline, linestyle="--", color="gray", label="Zero steering")
score_ax.set(title="Validation roads", xlabel="Epoch", ylabel="Validation MAE")
score_ax.legend()
fig.savefig(out / "experiment.png", dpi=150)
plt.show()